# PYNQ tANS na ZedBoard

Ten notebook pokazuje, jak wgrać bitstream FPGA oraz jak sterować jednostką tANS za pomoc AXI Lite.

mapa rejestrów:
- `0x00` — kontrola (bit0 = start, bit1 = wyczy flag done)
- `0x04` — status (bit0 = busy, bit1 = done sticky)
- `0x08` — symbol wejciowy
- `0x0C` — stan wejciowy
- `0x10` — stan wyjciowy
- `0x14` — wyjciowe bity
- `0x18` — liczba wyjciowych bitów
- `0x1000` — pocztek pamici tabeli tANS

In [ ]:
from pathlib import Path
from pynq import Overlay, MMIO

BITFILE_CANDIDATES = [
    Path('/home/xilinx/design_tans_wrapper.bit'),
    Path('/home/xilinx/ANS_prj/vitis_workspace_clean/ans_platform/hw/design_tans_wrapper.bit'),
    Path('design_tans_wrapper.bit')
]

BITFILE = next((p for p in BITFILE_CANDIDATES if p.exists()), None)
if BITFILE is None:
    raise FileNotFoundError(
        'Plik design_tans_wrapper.bit nie odnaleziony. Skopiuj go na pytk PYNQ, np. do /home/xilinx/.')

# Adresy rejestrów AXI Lite
ADDR_CTRL = 0x00
ADDR_STATUS = 0x04
ADDR_SYMBOL = 0x08
ADDR_STATE_IN = 0x0C
ADDR_STATE_OUT = 0x10
ADDR_BITS_OUT = 0x14
ADDR_NBITS_OUT = 0x18
TABLE_BASE_OFFSET = 0x1000
TABLE_ENTRIES = 1024
TABLE_BYTES = TABLE_ENTRIES * 4

# Adres bazowy IP w systemie Zynq
BASE_ADDRESS = 0x40000000

print('Uywam bitstreamu:', BITFILE)
print('Adres bazowy IP:', hex(BASE_ADDRESS))
print('Adres tabeli tANS:', hex(TABLE_BASE_OFFSET))

In [ ]:
class TansAxiLite:
    def __init__(self, bitfile, base_addr=BASE_ADDRESS, table_base=TABLE_BASE_OFFSET, table_entries=TABLE_ENTRIES):
        self.bitfile = Path(bitfile)
        self.base_addr = base_addr
        self.table_base = table_base
        self.table_entries = table_entries
        self.overlay = Overlay(str(self.bitfile))
        self.overlay.download()
        self.mmio = MMIO(self.base_addr, self.table_base + self.table_entries * 4)

    def write_reg(self, offset, value):
        self.mmio.write(offset, value)

    def read_reg(self, offset):
        return self.mmio.read(offset)

    def clear_done(self):
        self.write_reg(ADDR_CTRL, 0x2)

    def write_symbol(self, symbol):
        self.write_reg(ADDR_SYMBOL, int(symbol) & 0xFF)

    def write_state(self, state):
        self.write_reg(ADDR_STATE_IN, int(state) & 0xFFFF)

    def start(self, symbol, state_in=0):
        self.clear_done()
        self.write_symbol(symbol)
        self.write_state(state_in)
        self.write_reg(ADDR_CTRL, 0x1)
        return self.wait_done()

    def wait_done(self, timeout_s=1.0):
        import time
        start = time.time()
        while True:
            status = self.read_reg(ADDR_STATUS)
            busy = status & 0x1
            done = (status >> 1) & 0x1
            if not busy and done:
                return self.read_output()
            if time.time() - start > timeout_s:
                raise TimeoutError('Czekanie na zakończenie operacji tANS przekroczyło limit.')
            time.sleep(0.001)

    def read_output(self):
        return {
            'state_out': self.read_reg(ADDR_STATE_OUT) & 0xFFFF,
            'bits_out': self.read_reg(ADDR_BITS_OUT) & 0xFFFFFFFF,
            'nbits_out': self.read_reg(ADDR_NBITS_OUT) & 0x1F,
            'status': self.read_reg(ADDR_STATUS)
        }

    def write_table_entry(self, index, value):
        if index < 0 or index >= self.table_entries:
            raise IndexError('Indeks tabeli poza zakresem')
        addr = self.table_base + (index * 4)
        self.write_reg(addr, int(value) & 0xFFFFFFFF)

    def read_table_entry(self, index):
        if index < 0 or index >= self.table_entries:
            raise IndexError('Indeks tabeli poza zakresem')
        addr = self.table_base + (index * 4)
        return self.read_reg(addr)

    def load_table(self, values):
        if len(values) != self.table_entries:
            raise ValueError(f'Lista musi mieć długość {self.table_entries}')
        for idx, entry in enumerate(values):
            self.write_table_entry(idx, entry)

    def load_identity_table(self):
        values = []
        for idx in range(self.table_entries):
            next_state = (idx + 1) & 0xFFFF
            bits_out = 0
            nbits_out = 0
            values.append((next_state << 16) | (bits_out << 8) | nbits_out)
        self.load_table(values)

    def dump_status(self):
        status = self.read_reg(ADDR_STATUS)
        return {
            'busy': bool(status & 0x1),
            'done': bool((status >> 1) & 0x1),
            'raw': status
        }

    def show_registers(self):
        return {
            'CTRL': hex(self.read_reg(ADDR_CTRL)),
            'STATUS': hex(self.read_reg(ADDR_STATUS)),
            'SYMBOL': hex(self.read_reg(ADDR_SYMBOL)),
            'STATE_IN': hex(self.read_reg(ADDR_STATE_IN)),
            'STATE_OUT': hex(self.read_reg(ADDR_STATE_OUT)),
            'BITS_OUT': hex(self.read_reg(ADDR_BITS_OUT)),
            'NBITS_OUT': hex(self.read_reg(ADDR_NBITS_OUT))
        }

In [ ]:
tans = TansAxiLite(BITFILE)
print('Overlay załadowany:', BITFILE)
print('Status początkowy:', tans.dump_status())
print('Rejestry początkowe:')
print(tans.show_registers())

In [ ]:
tans.load_identity_table()
print('Tabela testowa wczytana. Przykad wpisu 0:', hex(tans.read_table_entry(0)))

In [ ]:
symbol = 5
state_in = 0
print('Startujemy kodowanie symbolu', symbol, 'ze stanem', state_in)
result = tans.start(symbol, state_in)
print('Wynik operacji:')
print(result)